# 01: Data Preparation untuk AI Model
**Tim CC26-PSU164** — Fair Price Finder for Freelancers

## Tujuan
Load dataset dari Data Science team, lakukan preprocessing untuk siap di-train.

## Workflow
1. Setup & Load Dataset v2.1
2. Define Features & Target
3. Train/Val/Test Split (Stratified)
4. Feature Scaling
5. Save Artifacts

In [15]:
from pathlib import Path
import os


def find_project_root():
    current_dir = Path.cwd().resolve()
    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / "data").exists():
            return candidate
    return current_dir


PROJECT_ROOT = find_project_root()
INPUT_PATH = PROJECT_ROOT / "data" / "output" / "dataset_v2.1_features.csv"
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "preprocessed"

os.makedirs(PREPROCESSED_DIR, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Input:  {INPUT_PATH}')
print(f'Output: {PREPROCESSED_DIR}')
print(f'Input exists: {INPUT_PATH.exists()}')

Project root: C:\laragon\www\fair-price-finder
Input:  C:\laragon\www\fair-price-finder\data\output\dataset_v2.1_features.csv
Output: C:\laragon\www\fair-price-finder\data\preprocessed
Input exists: True


In [16]:
import numpy as np
import pandas as pd
import pickle, json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Libraries loaded')

Libraries loaded


In [17]:
# LOAD DATASET
df = pd.read_csv(INPUT_PATH)

print(f'Shape: {df.shape}')
print(f'NaN: {df.isna().sum().sum()}')
print(f'Duplicates: {df.drop(columns=["row_id"]).duplicated().sum()}')
print(f'\nTarget distribution:')
print(df['log_price_single'].describe().round(3))

Shape: (4551, 49)
NaN: 0
Duplicates: 142

Target distribution:
count    4551.000
mean       12.690
std         1.032
min        10.820
25%        11.918
50%        12.612
75%        13.385
max        15.687
Name: log_price_single, dtype: float64


In [18]:
# DEFINE FEATURES & TARGET
TARGET_COL = 'log_price_single'
EXCLUDE_COLS = ['row_id', 'price_single', 'price_min', 'price_max',
                'log_price_single', 'log_price_min', 'log_price_max']
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

print(f'Target: {TARGET_COL}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'X shape: {X.shape}, y shape: {y.shape}')

Target: log_price_single
Features: 42
X shape: (4551, 42), y shape: (4551,)


In [19]:
# STRATIFIED SPLIT (70/15/15)
platform_cols = [c for c in df.columns if c.startswith('platform_')]
platform_label = df[platform_cols].idxmax(axis=1).values

# Split: 85% train+val, 15% test
X_trainval, X_test, y_trainval, y_test, plat_tv, plat_test = train_test_split(
    X, y, platform_label, test_size=0.15, stratify=platform_label, random_state=RANDOM_SEED
)

# Split: 70% train, 15% val (dari 85% sisa)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765, stratify=plat_tv, random_state=RANDOM_SEED
)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

Train: 3,185 | Val: 683 | Test: 683


In [20]:
# FEATURE SCALING (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f'Scaled. Train mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}')

Scaled. Train mean: 0.0000, std: 1.0000


In [21]:
# SAVE ALL ARTIFACTS
np.save(f'{PREPROCESSED_DIR}/X_train_scaled.npy', X_train_scaled)
np.save(f'{PREPROCESSED_DIR}/X_val_scaled.npy', X_val_scaled)
np.save(f'{PREPROCESSED_DIR}/X_test_scaled.npy', X_test_scaled)
np.save(f'{PREPROCESSED_DIR}/y_train.npy', y_train)
np.save(f'{PREPROCESSED_DIR}/y_val.npy', y_val)
np.save(f'{PREPROCESSED_DIR}/y_test.npy', y_test)

with open(f'{PREPROCESSED_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(f'{PREPROCESSED_DIR}/feature_names.pkl', 'wb') as f:
    pickle.dump(FEATURE_COLS, f)

metadata = {
    'dataset_version': 'v2.1',
    'target_column': TARGET_COL,
    'n_features': len(FEATURE_COLS),
    'n_train': len(X_train), 'n_val': len(X_val), 'n_test': len(X_test),
    'random_seed': RANDOM_SEED
}
with open(f'{PREPROCESSED_DIR}/preprocessing_info.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved 9 artifacts to:', PREPROCESSED_DIR)
for f in sorted(os.listdir(PREPROCESSED_DIR)):
    print(f'   {f}')

Saved 9 artifacts to: C:\laragon\www\fair-price-finder\data\preprocessed
   X_test_scaled.npy
   X_train_scaled.npy
   X_val_scaled.npy
   feature_names.pkl
   preprocessing_info.json
   scaler.pkl
   y_test.npy
   y_train.npy
   y_val.npy
